# Получение информации по поверке средств измерения в системе "Аршин" через API

Данный гайд показывает как можно получить информацию из системы "Аршин" и проверить информацию по поверке средств измерения используя стандартные библиотеки для работы с данными. Система имеет OEI-API.

OEI-API (Application Programming Interface) - программный интерфейс, предназначенный для предоставления в автоматическом режиме сведений о результатах поверок СИ, содержащихся в Федеральном информационном фонде по обеспечению единства измерений.

API обеспечивает возможность формирования и передачу запроса, и последующее получение результатов запроса в формате JSON. Данная возможность обеспечивается путем предоставления доступа к синхронным интерфейсам с использованием протокола HTTP 1.1.

## Ограничения на этапе тестирования и отладки:

* Выдача ограничена 10000 записей за 1 запрос (LIMIT 10000), планируется отдавать до 3млн. записей
* Вывод данных возможен в виде php array, json или xml. Устанавливается параметром &export_type (1 - php array; 2 - json; 3 - xml)
* Принудительное время выполнения скрипта ограничено 5 минутами. Оптимизируйте запрос под Ваши потребности.
* Планируется возвращение данных в виде сжатого массива
* Регистр при вводе параметр значеняи не имеет
* Все параметры (кроме ?regkey=d37e5f9c2df49556a580b1c3dc8dcc7a), указанные ниже, являются необязательными. Допустимы любые их комбиации
* Параметр regkey=d37e5f9c2df49556a580b1c3dc8dcc7a - ключ доступа для тестирования и отладки. Полноценный рабочий regkey предоставляется бонусом при приобретении годовой подписки на аналитику и будет доступен в профиле пользователя.

Подключим необходимые модули

In [1]:
import pandas as pd
import requests

Зададим точку входа для получения данных

In [2]:
url = 'http://731163-cj72200.tmweb.ru/vri/'

Проверим, что система "Аршин" нам отвечает

In [3]:
r = requests.get(url)
r

<Response [200]>

## Параметры запроса
&export_type - тип выходного массива данных.
1 - php array; 2 - json (для кодирования использована стандартаня PHP функция json_encode(), для декодирования следует использовать json_decode()); 3 - xml. Пример вывода данных в xml формате
http://731163-cj72200.tmweb.ru/vri/?regkey=d37e5f9c2df49556a580b1c3dc8dcc7a&vri_id_from=147222222&vri_id_to=147222322&export_type=3

&vri_id - id поверки.
Число в конце ccылки на карточку поверки в ФГИС АРШИН. Например для поверки https://fgis.gost.ru/fundmetrology/cm/results/1-179725564 это число 179725564

&vri_id_from - id поверки от.
Нижняя граница для поиска поверок по id. Например, при &vri_id_from=12345 будут выводиться только те поверки, чей id больше или равен 12345

&vri_id_to - id поверки до.
Верхняя граница для поиска поверок по id. Например, при &vri_id_to=99999 будут выводиться только те поверки, чей id меньше или равен 99999

&mi_number - Заводской номер СИ.
Текствое поле. Ищется строгое совпадение. Например, для поиска заводсого номера '05359326' необходимо задать &mi_number=05359326

&mi_modification - Модификация СИ.
Текствое поле. Ищется строгое совпадение. Например, для поиска модификации СИ 'Меркурий 230 ART-03 PQRSIDN' необходимо задать &mi_modification=Меркурий 230 ART-03 PQRSIDN

&mitypeTitle - Наименование СИ.
Текствое поле. Можно указывать часть фразы. Например, для поиска наименования СИ 'Счетчики электрической энергии трёхфазные статические' можно задать &mitypeTitle=электрической энергии

&mit_MPISI - Межповерочный интервал в соотвествии с ОТ.
Текствое поле. Можно указывать часть фразы. Например, для поиска такой фразы '4 года - для гор.воды; 6 лет - хол.' можно задать &mit_MPISI=6 лет - хол

&mit_id - id типа СИ в реестре СИ АРШИН.
Поле типа int. Ищется полное совпадение. Например: &mit_id=347162

&mitypeType - Обозначение типа СИ.
Текствое поле. Можно указывать часть фразы. Например, для поиска обозначения типа СИ 'Меркурий 230' можно задать &mitypeType=меркурий

&mitypeNumber - № типа СИ в госреестре.
Текствое поле. Ищется полное совпадение. Например: &mitypeNumber=23345-07

&org_title - Наименование организации-поверителя.
Текствое поле. Можно указывать часть фразы. Например, для поиска организации 'ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ ЭНЕРТЕСТ(ООО ЭНЕРТЕСТ)' можно задать &org_title=энертест

&mi_manufactureYear - Год выпуска СИ.
Целое число. Ищется полное совпадение. Например: &mi_manufactureYear=2009

&mi_signCipher - Условный шифр знака поверки.
Текстовое поле. Ищется полное совпадение. Например: &mi_signCipher=ГЦН

&docTitle - Наиенование методики поверки
Текстовое поле. Можно указывать часть фразы. Например, для документа ГОСТ OIML R 76-1-2011: &docTitle=ГОСТ OIML R 76

&mi_Owner_name - Владелец СИ.
Текствое поле. Можно указывать часть фразы. Например, для поиска организации 'ООО Газпром трансгаз Ухта' можно задать &mi_Owner_name=ООО Газпром

&mit_owner_CountrySI - Страна производства.
Текстовое поле. Ищется полное совпадение. Например: &mit_owner_CountrySI=россия

&mit_owner_SettlementSI - Населенный пункт (производства).
Текствое поле. Можно указывать часть слова или словосочетания. Например, для поиска СИ, произведенных в Москве: &mit_owner_SettlementSI=моск

&mit_owner_ManufacturerSI - Производитель СИ.
Текствое поле. Можно указывать часть фразы. Например, для поиска организации 'ООО Спутник' достаточно задать &mit_owner_ManufacturerSI=Спутник

&poverka_valid_date - Поверка действительна до
Текствое поле. Указывается дата окончания поверки в формате d.m.Y (например - &poverka_valid_date=17.05.2021). Ищется полное совпадение.

&poverka_publication_date - Дата публикации.
Текствое поле. Указывается дата публикации в формате d.m.Y (например - &poverka_publication_date=17.05.2021). Ищется полное совпадение.

&poverka_verification_date - Дата поверки.
Текствое поле. Указывается дата поверки в формате d.m.Y (например - &poverka_verification_date=12.03.2020). Ищется полное совпадение.

&poverka_verification_month - Месяц поверки.
Текствое поле. Указывается месяц поверки с годом в формате m.Y (например - &poverka_verification_month=03.2020). Ищется полное совпадение.

&poverka_verification_year - Год поверки.
Текствое поле. Указывается год в формате Y (например - &poverka_verification_year=2020). Ищется полное совпадение.

&poverka_vriType - Тип поверки.
Числовое поле. 2 - периодическая; 1 - первичная; Например, при такой записи - &poverka_vriType=2 будут отображены только периодические поверки.

## Пример запроса
Давайте запросим все поверенные приборы в МАИ, которые были сделаны в России и поверены в 2021 году.

Запрос займет некоторое время, а также не забывайте указывать регистрационный ключ (ключ в примере получен как тестовый и может не содержать всей информации из реестра)

In [4]:
keys = {'regkey': 'd37e5f9c2df49556a580b1c3dc8dcc7a',
        'mi_Owner_name': 'Новосибирский авиационный завод',
        #'poverka_verification_year': '2024',
        'export_type': '2'}

r = requests.get(url, params=keys)
r

<Response [200]>

Прочитаем данные из запроса

In [5]:
try:
    json = r.json()
except ValueError:
    print("Oops!")

Переведем данные в удобный нам формат Pandas DataFrame

In [6]:
df = pd.DataFrame(json)
df

,id,vri_id,mi_number,mi_modification,mi_manufactureYear,mi_signCipher,mi_Owner_name,org_title,fsa_ral_regNumbers_regNumber,mitypeNumber,...,vriType,result_docnum,result_doc_type,additional_info,means_npe,means_uve,means_mieta,means_ses,means_mis,means_reagent
0,1,41942091,7727,Е6-24/1,None,Н,Филиа...,Запад...,RA.RU...,25405-08,...,Периодическая,C-Н/0...,Извещение о непригодности,,,,56598.14.3Р.00198402 56598-14 Магазины сопроти...,,2303-68 Киловольтметры электростатические (№53...,
1,2,43465375,5299,КИСС-03,2018,Н,Филиа...,Запад...,RA.RU...,20641-11,...,Периодическая,C-Н/0...,Извещение о непригодности,,,,54727.13.2Р.00117862 54727-13 Компараторы-кали...,,1162-58 Катушки электрического сопротивления и...,
2,3,42536520,12480,М244,1970,Н,Филиа...,Запад...,RA.RU...,2373-68,...,Периодическая,C-Н/0...,Извещение о непригодности,,,,55804.13.1Р.00108519 55804-13 Калибраторы мног...,,,
3,4,38122750,1401,нет м...,2014,Н,Новос...,Запад...,RA.RU...,47965-11,...,Периодическая,C-Н/1...,Извещение о непригодности,,,3.1.ZZН.0046.2012 ГЭЕ длины 1 разряда в диапаз...,,,,
4,5,45933349,651358,КО-1,None,Н,Новос...,Запад...,RA.RU...,868-72,...,Периодическая,C-Н/1...,Извещение о непригодности,,,3.1.ZZН.0050.2013 ГЭЕ плоского угла 2 разряда ...,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,97,354894349,24211,Сейтр...,None,ВМ,ПЕРВИ...,ФЕДЕР...,RA.RU...,27033-13,...,Периодическая,C-ВМ/...,Извещение о непригодности,,,,46835.11.1Р.99495 46835-11 Меры профильные; 46...,,27015-04 Комплекты поверки гирь и весов перено...,
97,98,354964012,24210,Сейтр...,None,ВМ,ПЕРВИ...,ФЕДЕР...,RA.RU...,27033-13,...,Периодическая,C-ВМ/...,Извещение о непригодности,,,,46835.11.1Р.99495 46835-11 Меры профильные; 46...,,27015-04 Комплекты поверки гирь и весов перено...,
98,99,355973023,24210,Сейтр...,None,ВМ,ПЕРВИ...,ФЕДЕР...,RA.RU...,27033-13,...,Первичная поверка,C-ВМ/...,Извещение о непригодности,,,,46835.11.1Р.99495 46835-11 Меры профильные; 46...,,27015-04 Комплекты поверки гирь и весов перено...,
99,100,355973024,24211,Сейтр...,None,ВМ,ПЕРВИ...,ФЕДЕР...,RA.RU...,27033-13,...,Первичная поверка,C-ВМ/...,Извещение о непригодности,,,,46835.11.1Р.99495 46835-11 Меры профильные; 46...,,27015-04 Комплекты поверки гирь и весов перено...,


In [7]:
df['mitypeURL'][66]

'https://fgis.gost.ru/fundmetrology/registry/4/items/345372'

# Практическое задания
* Попробуйте получить данные о манометрах, поверенных в ЦАГИ имени Н.Е. Жуковского
* Проверьте есть ли в МАИ поверенные расходомеры
* Проверьте свой домашний счетчик воды (горячей или холодной) на наличие поверки, если конечно счетчик у вас установлен &#x1F600; и он был поверен после 24.09.2020 года (именно с этой даты все организации обязаны передавать данные о поверке в единую систему).

Все данные по п.1/2 обработайте и сведите в Pandas Dataframe в формат удобный для датасета по оценке запросов на поверку средств измерения

Постройте аналитику по полученным данным.

## Получение данных о манометрах, поверенных в ЦАГИ


In [8]:
csagi_manometers = {
    'regkey': 'd37e5f9c2df49556a580b1c3dc8dcc7a',
    'mitypeTitle': 'манометр',
    'org_title': 'ЦАГИ',
    'export_type': '2'
}

Отправка запроса

In [9]:
r = requests.get(url, params=csagi_manometers)

print("Статус:", r.status_code)

Статус: 200


In [10]:
try:
    data_manometers = r.json()
    df_manometers = pd.DataFrame(data_manometers)

except ValueError:
    print("Ошибка получения JSON")

In [11]:
df_manometers

,id,vri_id,mi_number,mi_modification,mi_manufactureYear,mi_signCipher,mi_Owner_name,org_title,fsa_ral_regNumbers_regNumber,mitypeNumber,...,vriType,result_docnum,result_doc_type,additional_info,means_npe,means_uve,means_mieta,means_ses,means_mis,means_reagent
0,1,1173224226,01-К,,None,АОЛ,,ФГУП ...,РОСС ...,55372-13,...,Без статуса,нет д...,Извещение о непригодности,,,3.1.АОЛ.0080.2016 Государственный эталон едини...,,,,
1,2,1173224227,02-К,,None,АОЛ,,ФГУП ...,РОСС ...,55372-13,...,Без статуса,нет д...,Извещение о непригодности,,,3.1.АОЛ.0080.2016 Государственный эталон едини...,,,,
2,3,1173224228,03-К,,None,АОЛ,,ФГУП ...,РОСС ...,55372-13,...,Без статуса,нет д...,Извещение о непригодности,,,3.1.АОЛ.0080.2016 Государственный эталон едини...,,,,
3,4,1173224229,04-К,,None,АОЛ,,ФГУП ...,РОСС ...,55372-13,...,Без статуса,нет д...,Извещение о непригодности,,,3.1.АОЛ.0080.2016 Государственный эталон едини...,,,,
4,5,1173224230,05-К,,None,АОЛ,,ФГУП ...,РОСС ...,55372-13,...,Без статуса,нет д...,Извещение о непригодности,,,3.1.АОЛ.0080.2016 Государственный эталон едини...,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5805,5806,135201421,131,МТ (0...,None,АОЛ,ФГУП ...,ФЕДЕР...,РОСС ...,3383-72,...,Периодическая,C-АОЛ...,Извещение о непригодности,,,,16026.97.2Р.74146 16026-97 Манометры избыточно...,,,
5806,5807,135201422,84,МТ (0...,None,АОЛ,ФГУП ...,ФЕДЕР...,РОСС ...,3383-72,...,Периодическая,C-АОЛ...,Извещение о непригодности,,,,16026.97.2Р.74146 16026-97 Манометры избыточно...,,,
5807,5808,135201423,17,МТ (0...,None,АОЛ,ФГУП ...,ФЕДЕР...,РОСС ...,3383-72,...,Периодическая,C-АОЛ...,Извещение о непригодности,,,,16026.97.2Р.74146 16026-97 Манометры избыточно...,,,
5808,5809,135201424,85,МТ (0...,None,АОЛ,ФГУП ...,ФЕДЕР...,РОСС ...,3383-72,...,Периодическая,C-АОЛ...,Извещение о непригодности,,,,16026.97.2Р.74146 16026-97 Манометры избыточно...,,,


In [12]:
print(f"Кол-во манометров, поверенных в ЦАГИ имени Н.Е. Жуковского", len(df_manometers))

Кол-во манометров, поверенных в ЦАГИ имени Н.Е. Жуковского 5810


## Проверка наличия поверенных расходомеров в МАИ

In [13]:
params_flowmeters = {
    'regkey': 'd37e5f9c2df49556a580b1c3dc8dcc7a',
    'mitypeTitle': 'расходомер',
    'mi_Owner_name': 'МАИ',
    'export_type': '2'
}

Выполнение запроса

In [15]:
r = requests.get(url, params=params_flowmeters)

print("Статус:", r.status_code)

Статус: 200


In [16]:
try:
    data_flowmeters = r.json()
    df_flowmeters = pd.DataFrame(data_flowmeters)

except ValueError:
    print("Ошибка получения JSON")

In [17]:
df_flowmeters

,id,vri_id,mi_number,mi_modification,mi_manufactureYear,mi_signCipher,mi_Owner_name,org_title,fsa_ral_regNumbers_regNumber,mitypeNumber,...,vriType,result_docnum,result_doc_type,additional_info,means_npe,means_uve,means_mieta,means_ses,means_mis,means_reagent
0,1,427990506,M1520...,EL-FLOW,2015,ДШЛ,НИИ П...,ОБЩЕС...,RA.RU...,25705-10,...,Периодическая,C-ДШЛ...,Извещение о непригодности,,,,40432.09.1Р.00334741 40432-09 Стенд для калибр...,,,
1,2,427990697,M1520...,EL-FLOW,2015,ДШЛ,НИИ П...,ОБЩЕС...,RA.RU...,25705-10,...,Периодическая,C-ДШЛ...,Извещение о непригодности,,,,40432.09.1Р.00334741 40432-09 Стенд для калибр...,,,
2,3,427990440,M1721...,EL-FLOW,2017,ДШЛ,НИИ П...,ОБЩЕС...,RA.RU...,64700-16,...,Периодическая,C-ДШЛ...,Извещение о непригодности,,,,40432.09.1Р.00334741 40432-09 Стенд для калибр...,,,
3,4,427990539,M1721...,EL-FLOW,2017,ДШЛ,НИИ П...,ОБЩЕС...,RA.RU...,64700-16,...,Периодическая,C-ДШЛ...,Извещение о непригодности,,,,40432.09.1Р.00334741 40432-09 Стенд для калибр...,,,
4,5,172632977,161954,Питер...,None,БЯ,МАИ+3Н,ФЕДЕР...,RA.RU...,46814-11,...,Периодическая,C-БЯ/...,Извещение о непригодности,"0,25 л/имп",,,53155.13.2Р.00163517 53155-13 Установки пролив...,,,
5,6,172632978,154478,Питер...,None,БЯ,МАИ+3Н,ФЕДЕР...,RA.RU...,46814-11,...,Периодическая,C-БЯ/...,Извещение о непригодности,"0,25 л/имп",,,53155.13.2Р.00163517 53155-13 Установки пролив...,,,
6,7,172632979,153857,Питер...,None,БЯ,МАИ+3Н,ФЕДЕР...,RA.RU...,46814-11,...,Периодическая,C-БЯ/...,Извещение о непригодности,"0,25 л/имп",,,53155.13.2Р.00163517 53155-13 Установки пролив...,,,
7,8,172632981,164427,Питер...,None,БЯ,МАИ+3Н,ФЕДЕР...,RA.RU...,46814-11,...,Периодическая,C-БЯ/...,Извещение о непригодности,"0,25 л/имп",,,53155.13.2Р.00163517 53155-13 Установки пролив...,,,
8,9,172632983,154157,Питер...,None,БЯ,МАИ+3Н,ФЕДЕР...,RA.RU...,46814-11,...,Периодическая,C-БЯ/...,Извещение о непригодности,"0,25 л/имп",,,53155.13.2Р.00163517 53155-13 Установки пролив...,,,
9,10,172632984,161095,Питер...,None,БЯ,МАИ+3Н,ФЕДЕР...,RA.RU...,46814-11,...,Периодическая,C-БЯ/...,Извещение о непригодности,"0,25 л/имп",,,53155.13.2Р.00163517 53155-13 Установки пролив...,,,


In [18]:
print(f"Кол-во расходомеров, поверенных в МАИ", len(df_flowmeters))

Кол-во расходомеров, поверенных в МАИ 21


## Проверка домашнего счетчика воды

In [49]:
params_ = {
    'regkey': 'd37e5f9c2df49556a580b1c3dc8dcc7a',
    'mitypeTitle': 'воды',
    'mitypeType': 'Пульсар',
    'mi_number': '11485359',
    'export_type': '2'
}

In [50]:
r = requests.get(url, params=params_)

print("Статус:", r.status_code)

Статус: 200


In [51]:
try:
    data_params = r.json()
    df_params = pd.DataFrame(data_params)

except ValueError:
    print("Ошибка получения JSON")

In [52]:
df_params

""


Проверить домашний счетчик воды не получилось((

### Подготовка данных в формат датасета, пригодный для оценки запросов на проверку

In [53]:
import pandas as pd


def normalize_dataset(
    df: pd.DataFrame,
    query_name: str,
    query_params: dict
) -> pd.DataFrame:
    base_columns = [
        'query_name', 'query_params', 'result_count',
        'vri_id', 'mi_number', 'mi_modification',
        'mitypeTitle', 'mitypeType', 'mitypeNumber',
        'org_title', 'mi_Owner_name',
        'verification_date', 'valid_date',
        'result', 'mitypeURL'
    ]

    if df.empty:
        return pd.DataFrame(columns=base_columns)

    out = df.copy()

    rename_map = {
        'poverka_verification_date': 'verification_date',
        'poverka_valid_date': 'valid_date',
        'verification_date': 'verification_date',
        'valid_date': 'valid_date',
        'applicable_until': 'valid_date',
    }
    out = out.rename(columns=rename_map)

    required_columns = [
        'vri_id', 'mi_number', 'mi_modification',
        'mitypeTitle', 'mitypeType', 'mitypeNumber',
        'org_title', 'mi_Owner_name',
        'verification_date', 'valid_date',
        'result', 'mitypeURL'
    ]
    for col in required_columns:
        if col not in out.columns:
            out[col] = pd.NA

    out['query_name'] = query_name
    out['query_params'] = str(query_params)
    out['result_count'] = len(out)

    for col in base_columns:
        if col not in out.columns:
            out[col] = pd.NA

    other_cols = [c for c in out.columns if c not in base_columns]
    return out[base_columns + other_cols]

In [55]:
csagi_manometers_data = normalize_dataset(df_manometers, "csagi_manometers", csagi_manometers)

flowmeters_data = normalize_dataset(df_flowmeters, "flowmeters", params_flowmeters)

dataset = pd.concat([csagi_manometers_data, flowmeters_data], ignore_index=True)
dataset.head()

,query_name,query_params,result_count,vri_id,mi_number,mi_modification,mitypeTitle,mitypeType,mitypeNumber,org_title,...,vriType,result_docnum,result_doc_type,additional_info,means_npe,means_uve,means_mieta,means_ses,means_mis,means_reagent
0,csagi_manometers,"{'regkey': 'd37e5f9c2df49556a580b1c3dc8dcc7a',...",5810,1173224226,01-К,,Маном...,P1454,55372-13,ФГУП ...,...,Без статуса,нет д...,Извещение о непригодности,,,3.1.АОЛ.0080.2016 Государственный эталон едини...,,,,
1,csagi_manometers,"{'regkey': 'd37e5f9c2df49556a580b1c3dc8dcc7a',...",5810,1173224227,02-К,,Маном...,P1454,55372-13,ФГУП ...,...,Без статуса,нет д...,Извещение о непригодности,,,3.1.АОЛ.0080.2016 Государственный эталон едини...,,,,
2,csagi_manometers,"{'regkey': 'd37e5f9c2df49556a580b1c3dc8dcc7a',...",5810,1173224228,03-К,,Маном...,P1454,55372-13,ФГУП ...,...,Без статуса,нет д...,Извещение о непригодности,,,3.1.АОЛ.0080.2016 Государственный эталон едини...,,,,
3,csagi_manometers,"{'regkey': 'd37e5f9c2df49556a580b1c3dc8dcc7a',...",5810,1173224229,04-К,,Маном...,P1454,55372-13,ФГУП ...,...,Без статуса,нет д...,Извещение о непригодности,,,3.1.АОЛ.0080.2016 Государственный эталон едини...,,,,
4,csagi_manometers,"{'regkey': 'd37e5f9c2df49556a580b1c3dc8dcc7a',...",5810,1173224230,05-К,,Маном...,P1454,55372-13,ФГУП ...,...,Без статуса,нет д...,Извещение о непригодности,,,3.1.АОЛ.0080.2016 Государственный эталон едини...,,,,


In [56]:
# Сколько записей вернул каждый запрос

dataset.groupby('query_name')['vri_id'].count()

,vri_id
query_name,
csagi_manometers,5810
flowmeters,21


In [57]:
# Топ организаций

dataset['org_title'].value_counts().head(10)

,count
org_title,
ФЕДЕР...,3588
ФГУП ...,2235
ОБЩЕС...,6
Федер...,2


In [58]:
# Распределение по годам поверки

dataset['verification_date'] = pd.to_datetime(
    dataset['verification_date'],
    errors='coerce'
)

dataset['verification_date'].dt.year.value_counts().sort_index()

/tmp/ipykernel_11614/25453802.py:3: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  dataset['verification_date'] = pd.to_datetime(


,count
verification_date,
2017,805
2018,571
2019,728
2020,745
2021,789
2022,642
2023,570
2024,526
2025,419


In [59]:
# Популярные типы приборов

dataset['mitypeTitle'].value_counts().head(10)

,count
mitypeTitle,
Маном...,5803
Расхо...,20
Микро...,5
Тягом...,2
Счетч...,1
